# Edge TinyML — Recyclables Classifier (Training Notebook)
**Student:** Remmy Kipruto Tumo


In [ ]:
!pip install -q tensorflow==2.12.0

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers
import os
print('TF version:', tf.__version__)

In [ ]:
DATA_DIR = '/content/data'
IMG_SIZE = (160,160)
BATCH_SIZE = 32
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR+'/train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR+'/val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE+(3,), include_top=False, weights='imagenet')
base_model.trainable = False
inputs = tf.keras.Input(shape=IMG_SIZE+(3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_ds, epochs=10, validation_data=val_ds)

In [ ]:
model.save('model_recyclables.h5')

In [ ]:
def representative_data_gen():
    for images, labels in train_ds.take(50):
        yield [tf.cast(images, tf.float32).numpy()]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
open('model_recyclables_quant.tflite','wb').write(tflite_model)